In [1]:
def main(datasources, start_date, end_date):
    """多频率融合：1min/5min/15min/30min 四通道独立编码后融合。

    赛制约定: 平台只替换 datasources / start_date / end_date, 其中
    start_date~end_date 为【测试集区间】。训练区间写死(TRAIN_START/END),
    用样本外的测试区间做预测, 输出每日分数 ['date','instrument','score']。
    切勿用传入的 start_date/end_date 训练(数据泄漏, 会被审查)。

    训练读【写死的开发数据表】, 推理读【平台注入的】datasources。
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    import structlog

    logger = structlog.get_logger()

    # ---------- 配置 (写死, 不随平台入参变化) ----------
    TRAIN_TABLE_1M = "bigalpha_2026_stock_bar1m"
    # 推理用表从平台注入
    INFER_TABLE_1M = datasources["bar1m"]
    INFER_TABLE_5M = datasources.get("bar5m", "bigalpha_2026_stock_bar5m")
    INFER_TABLE_15M = datasources.get("bar15m", "bigalpha_2026_stock_bar15m")
    INFER_TABLE_30M = datasources.get("bar30m", "bigalpha_2026_stock_bar30m")

    TRAIN_START, TRAIN_END = "2022-01-01", "2023-12-31 23:59:59"
    SEQ_LEN_1M = 64
    EPOCHS, BATCH, LR, SEED = 5, 256, 1e-3, 42
    MAX_TRAIN_INSTRUMENTS = 200

    PRICE_COLS = ["close", "bid_price1", "ask_price1", "bid_price5", "ask_price5"]
    VOL_COLS = [
        "volume", "amount", "deal_number",
        "bid_volume1", "ask_volume1", "bid_volume5", "ask_volume5",
        "bid_num_orders1", "ask_num_orders1", "bid_num_orders5", "ask_num_orders5",
    ]
    FEATURE_COLS = PRICE_COLS + VOL_COLS
    N_FEAT = len(FEATURE_COLS)

    FREQ_CONFIG = {
        "1m":  ("bigalpha_2026_stock_bar1m",  64, 1),
        "5m":  ("bigalpha_2026_stock_bar5m",  13, 5),
        "15m": ("bigalpha_2026_stock_bar15m",  5, 15),
        "30m": ("bigalpha_2026_stock_bar30m",  3, 30),
    }

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- 位置编码 ----------
    def make_sinusoidal_pos_encoding(seq_len, d_model):
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)
                             * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    # ---------- 模型 ----------
    class FreqEncoder(nn.Module):
        def __init__(self, n_feat, seq_len, d_model=64, nhead=4, nlayers=1, dim_ff=128):
            super().__init__()
            self.proj = nn.Linear(n_feat, d_model)
            self.register_buffer("pos", make_sinusoidal_pos_encoding(seq_len, d_model))
            layer = nn.TransformerEncoderLayer(
                d_model, nhead, dim_ff, 0.1, batch_first=True, activation="gelu"
            )
            self.encoder = nn.TransformerEncoder(layer, nlayers)
            self.attn_pool = nn.MultiheadAttention(d_model, nhead, batch_first=True)
            self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        def forward(self, x):
            h = self.encoder(self.proj(x) + self.pos)
            B = h.shape[0]
            q = self.pool_query.expand(B, -1, -1)
            pooled, _ = self.attn_pool(q, h, h)
            return pooled.squeeze(1)

    class MultiFreqModel(nn.Module):
        def __init__(self, n_feat, d_encode=64, d_fusion=128, nhead=4):
            super().__init__()
            self.enc_1m  = FreqEncoder(n_feat, 64, d_encode, nhead, nlayers=1)
            self.enc_5m  = FreqEncoder(n_feat, 13, d_encode, nhead, nlayers=1)
            self.enc_15m = FreqEncoder(n_feat,  5, d_encode, nhead, nlayers=1)
            self.enc_30m = FreqEncoder(n_feat,  3, d_encode, nhead, nlayers=1)

            self.fusion = nn.Linear(4 * d_encode, d_fusion)
            self.head = nn.Sequential(
                nn.LayerNorm(d_fusion),
                nn.GELU(),
                nn.Linear(d_fusion, d_fusion // 2),
                nn.GELU(),
                nn.Linear(d_fusion // 2, 1),
            )

        def forward(self, x_list):
            h1  = self.enc_1m(x_list[0])
            h5  = self.enc_5m(x_list[1])
            h15 = self.enc_15m(x_list[2])
            h30 = self.enc_30m(x_list[3])
            h = self.fusion(torch.cat([h1, h5, h15, h30], dim=-1))
            return self.head(h).squeeze(-1)

    # ---------- 数据 ----------
    def build_freq_dataset(freq_table, sd, ed, instruments, seq_len, mode="train", stats=None):
        """单频率数据集构建。返回 (X, y, keys, stats)。"""
        import gc
        t0 = time.time()
        buf = (pd.to_datetime(sd) - pd.Timedelta(days=20)).strftime("%Y-%m-%d")
        sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)
        wins, ys, keys = [], [], []

        chunk_size = 50
        for ci in range(0, len(instruments), chunk_size):
            chunk = instruments[ci: ci + chunk_size]
            sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {freq_table} ORDER BY instrument, date"
            df = dai.query(sql, filters={"date": [buf, ed], "instrument": chunk}).df()
            for c in VOL_COLS:
                df[c] = np.log1p(df[c].clip(lower=0))

            for ins, sub in df.groupby("instrument", sort=False):
                if len(sub) <= seq_len:
                    continue
                feats = sub[FEATURE_COLS].to_numpy(np.float32)
                day = sub["date"].dt.normalize().to_numpy()
                close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))
                close_px = sub["close"].to_numpy(np.float64)[close_pos]
                dates = day[close_pos]
                for k, p in enumerate(close_pos):
                    d = pd.Timestamp(dates[k])
                    if p + 1 < seq_len or d < sd_ts or d > ed_ts:
                        continue
                    label = None
                    if k + 1 < len(close_pos) and close_px[k] > 0:
                        r = close_px[k + 1] / close_px[k] - 1.0
                        if np.isfinite(r):
                            label = np.float32(r)
                    if mode == "train" and label is None:
                        continue  # 训练集跳过无标签样本
                    wins.append(feats[p - seq_len + 1 : p + 1])
                    ys.append(label if label is not None else np.float32(0.0))
                    keys.append((d, ins))
            del df
            gc.collect()

        X = np.stack(wins).astype(np.float16)
        del wins; gc.collect()

        if stats is None:
            flat = X.reshape(-1, N_FEAT).astype(np.float32)
            stats = (flat.mean(0).astype(np.float32), flat.std(0).astype(np.float32) + 1e-6)
            del flat; gc.collect()

        m, s = stats
        X = ((X - m.astype(np.float16)) / s.astype(np.float16)).astype(np.float16)
        logger.info(f"  {freq_table} 构建完成", samples=len(keys), elapsed=round(time.time() - t0, 1))
        return X, np.array(ys, np.float32), keys, stats

    def pool(table, sd, ed):
        df = dai.query(f"SELECT DISTINCT instrument FROM {table}",
                       filters={"date": [sd, ed]}).df()
        return df["instrument"].tolist()

    # ---------- 训练 ----------
    instruments_train = pool(TRAIN_TABLE_1M, TRAIN_START, TRAIN_END)[:MAX_TRAIN_INSTRUMENTS]
    logger.info("训练标的数", n=len(instruments_train))
    logger.info("=== 构建训练集 ===")
    stats_all = {}
    X_train_list, y_all = [], None

    for freq_name, (freq_table, seq_len, _) in FREQ_CONFIG.items():
        X, y, keys, s = build_freq_dataset(
            freq_table, TRAIN_START, TRAIN_END, instruments_train, seq_len,
            mode="train"
        )
        X_train_list.append(X)
        stats_all[freq_name] = s
        if y_all is None:
            y_all = y
        else:
            assert len(y) == len(y_all), f"{freq_name} 样本数不一致"

    lo, hi = np.percentile(y_all, [1, 99])
    y_clipped = np.clip(y_all, lo, hi)

    model = MultiFreqModel(N_FEAT).to(device)
    logger.info("可训练参数量", n_params=sum(p.numel() for p in model.parameters()))

    class MultiFreqDataset(torch.utils.data.Dataset):
        def __init__(self, X_list, y):
            self.X_list = [torch.from_numpy(x) for x in X_list]
            self.y = torch.from_numpy(y)
        def __len__(self):
            return len(self.y)
        def __getitem__(self, idx):
            return tuple(x[idx] for x in self.X_list), self.y[idx]

    def collate_fn(batch):
        xs, ys = zip(*batch)
        n_freq = len(xs[0])
        xs_grouped = [torch.stack([x[i] for x in xs]) for i in range(n_freq)]
        return xs_grouped, torch.stack(ys)

    loader = DataLoader(
        MultiFreqDataset(X_train_list, y_clipped),
        batch_size=BATCH, shuffle=True, collate_fn=collate_fn,
        pin_memory=(device.type == "cuda"),
    )

    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    total_steps = EPOCHS * len(loader)
    warmup_steps = int(0.1 * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return max(0.5 * (1 + np.cos(np.pi * progress)), 1e-5 / LR)

    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    for ep in range(EPOCHS):
        model.train()
        tot_loss, nb = 0.0, 0
        total_batches = len(loader)
        log_interval = max(1, total_batches // 5)

        for bi, (xb_list, yb) in enumerate(loader):
            xb_list = [x.to(device, non_blocking=True).float() for x in xb_list]
            yb = yb.to(device, non_blocking=True)
            opt.zero_grad()
            loss = loss_fn(model(xb_list), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            scheduler.step()
            tot_loss += loss.item(); nb += 1

            if bi % log_interval == 0:
                logger.info(f"epoch {ep+1}/{EPOCHS} batch {bi+1}/{total_batches}",
                            loss=round(tot_loss / (nb or 1), 6))

        with torch.no_grad():
            model.eval()
            fp = model([x[:4096].to(device).float() for x in
                        [torch.from_numpy(x) for x in X_train_list]]).cpu().numpy()
            ic = np.corrcoef(fp, y_clipped[:4096])[0, 1]
        logger.info("epoch 完成", epoch=ep + 1, mse=round(tot_loss / max(nb, 1), 6),
                    train_ic=round(float(ic), 4))

    # ---------- 推理 (样本外测试区间, 用平台注入表) ----------
    logger.info("=== 推理 ===")
    instruments_infer = pool(INFER_TABLE_1M, start_date, end_date)

    Xte_list, idx_dfs = [], []
    for freq_name, (freq_table, seq_len, _) in FREQ_CONFIG.items():
        infer_table = {
            "1m": INFER_TABLE_1M, "5m": INFER_TABLE_5M,
            "15m": INFER_TABLE_15M, "30m": INFER_TABLE_30M,
        }[freq_name]
        X, _, keys, _ = build_freq_dataset(
            infer_table, start_date, end_date, instruments_infer, seq_len,
            mode="infer",
            stats=stats_all[freq_name]
        )
        Xte_list.append(torch.from_numpy(X))
        idx_dfs.append(keys)

    # 多频率取交集
    key_sets = [set(tuple(k) for k in kd) for kd in idx_dfs]
    common_keys = key_sets[0]
    for ks in key_sets[1:]:
        common_keys = common_keys & ks

    idx_df_0 = pd.DataFrame(idx_dfs[0], columns=["date", "instrument"])
    idx_map = {tuple(row): i for i, row in enumerate(idx_dfs[0])}
    common_indices = [idx_map[k] for k in common_keys if k in idx_map]

    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(common_indices), BATCH):
            batch_idx = common_indices[i: i + BATCH]
            xb_list = [x[batch_idx].to(device).float() for x in Xte_list]
            preds.append(model(xb_list).cpu().numpy())

    all_preds = np.concatenate(preds).astype(np.float64)
    result_df = idx_df_0.iloc[common_indices].copy()
    result_df["score"] = all_preds

    # ---------- 对齐中证 1000 + 规范输出 ----------
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(result_df, stk, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
                .reset_index(drop=True))
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
    }

    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)

    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        show=True,
    )

[2026-07-05 13:37:00] [info     ] 计算分数                           end='2024-12-31 23:59:59' start='2024-01-01 00:00:00'
[2026-07-05 13:37:10] [info     ] 训练标的数                          n=200
[2026-07-05 13:37:10] [info     ] === 构建训练集 ===
